In [1]:
import numpy as np
import torch
import pandas as pd
from helper import Autoencoder, load_data, train, save_params

In [2]:
known_strengths = {'null':10,'N4': 0.0, 'Q4': 1.3340727612197436, 'Q7': 2.428134794028789, 'T4': 1.9599578912997808, 'V4': 3.2307473950102388, 'G4': 4.514668716435611, 'E1': 5.21564553829087, 'A2': 0.43209185328878835, 'Y3': 0.8603530802315512}

In [3]:
ignore_indexes=['Event','Replicate']
group_size=100
batch_size=100
reconstruction_weight=1
strength_weight=0.0001
num_epochs=5
embedding_size=2
autoencoder_hidden_sizes=[250,200]
train_test_split=0.8
model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD38', 'CD4', 'CD44', 'CD45',
       'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
       'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
       'Proliferation', 'SSC-A', 'TBet']
# model_inputs = ['CD126', 'CD19', 'CD2', 'CD25', 'CD27', 'CD4', 'CD44', 'CD45', #try without CD38
#        'CD45RA', 'CD5', 'CD62L', 'CD86', 'CD8a', 'CX3CR1', 'CXCR6', 'FSC-A',
#        'Granzyme B', 'ICOS', 'IRF8', 'MHC-II', 'OX40', 'PD-L1',
#        'Proliferation', 'SSC-A', 'TBet']
model_path = "autoencoder_test.pt"
data_path = "../../initialSingleCellDf-channel-20220916-MW_018-001.h5"

In [4]:
data = pd.read_hdf(data_path, key="df")
data = data.loc[(data.index.get_level_values('CellType') == 'OT-1')]#& (data.index.get_level_values('Peptide') != 'T4') 

In [5]:
dataset, index_order, data_index_names, data_columns, data_labels,missing_columns = load_data(data,model_inputs,ignore_indexes,group_size)
print("Missing columns: ",missing_columns)

Missing columns:  []


In [6]:
# add to the dataset the known strengths for each sample
antigen_column = list(data_index_names).index('Peptide')
def get_strength(labels):
    antigen = labels[antigen_column]
    if antigen in known_strengths:
        return known_strengths[antigen]
    else:
        print("Antigen not found: ",antigen)
        return np.nan

def add_column_to_label(dataset,new_column):
    data = []
    labels = []
    for i in range(len(dataset)):
        new_labels = np.append(dataset[i][1],new_column[i])
        data.append(np.array(dataset[i][0]))
        labels.append(new_labels)
    data = np.array(data)
    labels = np.array(labels, dtype=np.float32)  # Convert labels to float32
    new_dataset = torch.utils.data.TensorDataset(torch.tensor(data),torch.tensor(labels))
    return new_dataset, len(labels[0])-1

strengths = [get_strength(labels) for labels in data_labels]
new_dataset, strength_index = add_column_to_label(dataset,strengths)

In [7]:
train_size = int(train_test_split * len(new_dataset))
test_size = len(new_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(new_dataset, [train_size, test_size])
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = Autoencoder(len(model_inputs),autoencoder_hidden_sizes,embedding_size).to(device)

In [9]:
train(model,train_loader,test_loader,reconstruction_weight,strength_weight,strength_index,device,num_epochs=num_epochs)
torch.save(model.state_dict(), model_path)
save_params(model_inputs,group_size,batch_size,embedding_size,autoencoder_hidden_sizes,model_path)

epoch [1/5], train loss:0.004397 val loss:0.001250
epoch [2/5], train loss:0.001128 val loss:0.001072
epoch [3/5], train loss:0.000970 val loss:0.000931
epoch [4/5], train loss:0.000899 val loss:0.000924
epoch [5/5], train loss:0.000855 val loss:0.000836
